> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 10 · Notebook 12 — RL agents under scrutiny, and optimal execution

**Sessions:** S23–S24 (RL strategies: trading, execution & hedging · RL optimization & M7a) · [Lesson plan](../../docs/lessons/PART_10_MACHINE_LEARNING.md) · graded labs in [`labs/part10/`](../../labs/part10/)

**You will:**
1. Write the Q-learning update and train an agent in the trading environment.
2. Judge it across seeds, regimes and costs, against simple rules.
3. Compute the Almgren–Chriss execution schedule and its cost–risk trade-off.
4. Check the schedule in a simulated execution environment.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known structure (a hidden regime, a known autocorrelation, planted importances, pure noise), so you can see what a model can learn, and that it learns nothing from noise.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p10lib.py is in notebooks/part10/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p10lib as p

p.use_course_style()

## 1. A Q-learning trader

Six states (was the last return up or down? × current position), three actions. After each step, move `Q[s, a]` toward the target `r + γ · max Q[s']`, or just `r` when the episode has ended: `Q[s, a] += α·(target − Q[s, a])`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def q_learning_env(env, n_states, state_fn=p.sign_state, episodes=200, alpha=0.05, gamma=0.9, eps=0.1, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((n_states, 3))
    for ep in range(episodes):
        obs, _ = env.reset(seed=seed + ep)
        s, done = state_fn(obs), False
        while not done:
            a = int(rng.integers(3)) if rng.random() < eps else int(Q[s].argmax())
            obs, r, done, _, _ = env.step(a)
            s2 = state_fn(obs)
            target = r if done else r + gamma * Q[s2].max()
            Q[s, a] += alpha * (target - Q[s, a])
            s = s2
    return Q

returns = p.ar1(4000, 0.3, 0.01, seed=0)
train, test = returns[:3000], returns[3000:]
train_env = p.TradingEnv(train, cost=0.0002, episode_len=250)          # random 250-day episodes
cases = [dict(episodes=40, seed=0), dict(episodes=40, seed=1, gamma=0.0)]
mine = [p.attempt(q_learning_env, train_env, 6, **c) for c in cases]
mine = p.check("q_learning_env", mine, [p.q_learning_env(train_env, 6, **c) for c in cases])
print("states: (last return > 0) × 3 + position + 1;  Q after 40 episodes:")
pd.DataFrame(mine[0], columns=["short", "flat", "long"], index=[f"{'up' if s >= 3 else 'down'} day, position {s % 3 - 1:+d}" for s in range(6)]).round(4)

## 2. The defense: seeds, regimes, costs

An RL result is only credible across **seeds** (the agent's randomness), **regimes** (momentum φ = +0.3, noise φ = 0, reversal φ = −0.3) and **costs**, next to the simple rules. Five agents per regime, 300 episodes each (about 10 seconds):

In [ ]:
def greedy(Q):
    return lambda o: int(Q[p.sign_state(o)].argmax())

rules = {"long only": lambda o: 2, "momentum": lambda o: 2 if o[-2] > 0 else 0, "reversal": lambda o: 0 if o[-2] > 0 else 2}
rows = {}
for phi in (0.3, 0.0, -0.3):
    r = p.ar1(4000, phi, 0.01, seed=0)
    tr, te = r[:3000], r[3000:]
    test_env = p.TradingEnv(te, cost=0.0002, scale=tr.std())
    agents = [p.run_policy(test_env, greedy(p.q_learning_env(p.TradingEnv(tr, cost=0.0002, episode_len=250), 6, episodes=300, seed=s)))
              for s in range(5)]
    rows[f"φ = {phi:+.1f}"] = {"agent min": min(agents), "agent mean": np.mean(agents), "agent max": max(agents)} | {
        k: p.run_policy(test_env, f) for k, f in rules.items()}
pd.DataFrame(rows).T.round(2)

With momentum, the agents' results range from a loss to the momentum rule's result, never above it: at best the agent rediscovers the one-line rule. On noise it learns to stay out; with reversal it learns reversal. That is what a working agent looks like on data with known structure, and why its seed spread must be reported.

Now train with **free** trades and test at a realistic 30 bp:

In [ ]:
r = p.ar1(4000, 0.3, 0.01, seed=0)
tr, te = r[:3000], r[3000:]
costly_test = p.TradingEnv(te, cost=0.003, scale=tr.std())
rows = {}
for s in range(3):
    free = p.q_learning_env(p.TradingEnv(tr, cost=0.0, episode_len=250), 6, episodes=300, seed=s)
    aware = p.q_learning_env(p.TradingEnv(tr, cost=0.003, episode_len=250), 6, episodes=300, seed=s)
    rows[f"seed {s}"] = {"trained with free trades": p.run_policy(costly_test, greedy(free)),
                         "trained with 30 bp costs": p.run_policy(costly_test, greedy(aware))}
print(f"momentum rule at 30 bp: {p.run_policy(costly_test, rules['momentum']):.2f}")
pd.DataFrame(rows).T.round(3)

Agents trained without costs trade every day and lose at real costs. Agents trained with costs learn that the edge doesn't pay for them and stay flat, which is the right answer: even the momentum rule loses at 30 bp.

## 3. Optimal execution

Selling `X` shares over `N` periods: fast selling costs market impact, slow selling risks the price moving. Almgren & Chriss (2000) minimize `E[cost] + λ·Var[cost]`. With `η̃ = η − γ/2` and `κ` from `cosh κ = 1 + λσ²/(2η̃)`, the holdings after `j` trades are `x_j = X·sinh(κ(N − j))/sinh(κN)`; the trades are the differences. As `λ → 0` it becomes TWAP.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def almgren_chriss_schedule(X, N, sigma, eta, gamma, lam):
    eta_t = eta - gamma / 2
    kappa = np.arccosh(1 + lam * sigma ** 2 / (2 * eta_t))
    if kappa < 1e-12:
        return p.twap_schedule(X, N)
    j = np.arange(N + 1)
    holdings = X * np.sinh(kappa * (N - j)) / np.sinh(kappa * N)
    return -np.diff(holdings)

X_SH, N, SIGMA, ETA, GAMMA = 1000, 20, 0.3, 0.01, 0.001
cases = [(X_SH, N, SIGMA, ETA, GAMMA, 1e-3), (X_SH, N, SIGMA, ETA, GAMMA, 1e-2), (500, 10, 0.5, 0.02, 0.0, 1e-4)]
mine = [p.attempt(almgren_chriss_schedule, *c) for c in cases]
mine = p.check("almgren_chriss_schedule", mine, [p.almgren_chriss_schedule(*c) for c in cases])
ac = mine[0]

In [ ]:
twap = p.twap_schedule(X_SH, N)
(e_tw, v_tw), (e_ac, v_ac) = p.cost_moments(twap, SIGMA, ETA, GAMMA), p.cost_moments(ac, SIGMA, ETA, GAMMA)
print(f"TWAP: expected cost {e_tw:.0f}, std {np.sqrt(v_tw):.0f};  Almgren–Chriss (λ = 1e-3): expected cost {e_ac:.0f}, std {np.sqrt(v_ac):.0f}")
print(f"→ {e_ac / e_tw - 1:+.1%} expected cost for {v_ac / v_tw - 1:+.1%} variance")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
axes[0].bar(np.arange(N) - 0.2, twap, 0.4, label="TWAP"); axes[0].bar(np.arange(N) + 0.2, ac, 0.4, label="Almgren–Chriss")
axes[0].legend(); axes[0].set_title("shares sold per period")
lams = np.geomspace(1e-6, 1e-1, 30)
frontier = np.array([p.cost_moments(p.almgren_chriss_schedule(X_SH, N, SIGMA, ETA, GAMMA, lam), SIGMA, ETA, GAMMA) for lam in lams])
axes[1].plot(np.sqrt(frontier[:, 1]), frontier[:, 0], marker=".")
axes[1].scatter([np.sqrt(v_tw)], [e_tw], color="#eb6834", zorder=3, label="TWAP")
axes[1].set_xlabel("std of cost"); axes[1].set_ylabel("expected cost"); axes[1].legend(); axes[1].set_title("the efficient frontier of schedules")
plt.tight_layout(); plt.show()

Almgren–Chriss front-loads the selling: it pays a little more expected cost to cut the variance by about a third. The frontier is the benchmark any RL execution agent must beat on its own objective, and a hard one to beat, because it is optimal for this model.

## 4. The execution environment

`p.ExecutionEnv` simulates the same model with random prices (action = the fraction of the remaining shares to sell now; total reward = −implementation shortfall). Run each schedule on 500 paths:

In [ ]:
env = p.ExecutionEnv(X_SH, N, SIGMA, ETA, GAMMA)
sims = {"TWAP": p.simulate_execution(env, p.schedule_policy(twap)), "Almgren–Chriss": p.simulate_execution(env, p.schedule_policy(ac)),
        "wait, dump at the end": p.simulate_execution(env, lambda obs, k: 0.0, 200),
        "dump everything now": p.simulate_execution(env, lambda obs, k: 1.0, 200)}
pd.DataFrame({k: {"mean shortfall": v.mean(), "std": v.std()} for k, v in sims.items()}).T.round(0)

The simulation agrees with the formulas within sampling error, and the naive extremes cost ten times more. An RL execution agent would be trained in this environment, and it passes only if it lands on (or beyond) the frontier above, across seeds.

## S24 · Tuning RL and the M7a release

Tuning an RL agent (learning rate, γ, ε schedule, state design, reward) is a search like any other: every configuration is a trial for the Deflated Sharpe Ratio, and each must be judged across seeds and regimes, as above. The milestone M7a release requires: the environment's timing and cost tests, the rule baselines, the seed/regime/cost table, and a clear statement of whether the agent beats the best rule. Clinic W6 in the labs builds that defense (optionally with PPO from Stable-Baselines3).

## Wrap-up

* An RL agent is credible only across seeds, regimes and costs, next to simple rules.
* Train with realistic costs; free trades teach over-trading.
* In execution, Almgren–Chriss defines the frontier an agent must reach.
* Graded version: `labs/part10/week38_rl` (`q_learning_env`, `almgren_chriss_schedule`, `cost_moments`, `ExecutionEnv`) and Clinic W6 (the RL defense).